# Lab 3 — Scale and encode: preparing data for a model
**Data Analysis — Master AI / Master RSD** · Chapter 2: sections 6 (Normalisation) and 7 (Encoding)

*Put variables on the same scale, turn text into numbers, without introducing artificial relationships.*

**Lab objectives**
1. **Identify the nature** of each variable (quantitative, nominal, ordinal, target).
2. **Scale** the numeric variables: Z-score, Min-Max, log transform → *choose the suitable method*.
3. **Encode** the categorical variables: One-Hot, Ordinal, Label → *avoid the classic pitfalls*.
4. **Assemble** a reproducible preparation (`ColumnTransformer`) and **measure its effect** on a k-NN model.
5. **Document** every decision in this notebook.

| Duration | Organisation | Data | Deliverable |
|---|---|---|---|
| 2 sessions of 2 h (4 h total) | Pairs · Jupyter notebook | `clients_enriched.csv` (fictional, provided) | Completed notebook + conclusions |

⭐ **Continuity:** `clients_enriched.csv` is the **cleaned file from Lab 2** (988 clients, no missing values), enriched with three categorical variables: `city`, `education`, `satisfaction`.

**How to use**: complete the cells marked `# TODO`, then run the **✅ Self-check** cell that follows. Answer the questions in the **✍️** cells.

---
## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)

---
## Step 1 — Identify the nature of each variable
*The method depends on the nature **and** the role of the variable* (section 7, good practice ①).

### 1.0 Load the file

In [ ]:
df = ...   # TODO: pd.read_csv(...) — separator ";"

print(df.shape)
df.head()

✅ **Self-check**

In [ ]:
assert df.shape == (988, 11), f"Expected (988, 11), got {df.shape} -> separator?"
assert df.isna().sum().sum() == 0, "the file is already clean: no missing values"
print("✅ 1.0 correct")

### 1.1 Classify the variables
We want to **predict the client's `segment`** (A, B or C) from the other variables.
For each column, give its nature among: `"identifier"`, `"quantitative"`, `"nominal"`, `"ordinal"`, `"target"`.

💡 Look at the categories with `df["column"].unique()` if in doubt.

In [ ]:
for col in ["city", "education", "satisfaction", "segment"]:
    print(f"{col:13s}", sorted(df[col].unique()))

In [ ]:
nature = {
    "id":           ...,   # TODO
    "age":          ...,
    "income":       ...,
    "n_children":   ...,
    "tenure":       ...,
    "n_purchases":  ...,
    "city":         ...,
    "education":    ...,
    "satisfaction": ...,
    "email":        ...,
    "segment":      ...,
}

✅ **Self-check**

In [ ]:
assert nature["city"] == "nominal", "is there a natural order between cities?"
assert nature["education"] == nature["satisfaction"] == "ordinal"
assert nature["segment"] == "target"
assert nature["id"] == nature["email"] == "identifier"
assert all(nature[c] == "quantitative" for c in ["age", "income", "n_children", "tenure", "n_purchases"])
print("✅ 1.1 correct")

In [ ]:
# These lists are used throughout the lab
num_cols = [c for c, t in nature.items() if t == "quantitative"]
nom_cols = [c for c, t in nature.items() if t == "nominal"]
ord_cols = [c for c, t in nature.items() if t == "ordinal"]
print("numeric:", num_cols)
print("nominal:", nom_cols)
print("ordinal:", ord_cols)

---
## Step 2 — Scaling the numeric variables (section 6)

### 2.1 The problem: different scales
Display the descriptive statistics of the numeric variables. Compare the **range** of `age` and `income`.

In [ ]:
# TODO: descriptive statistics of num_cols

range_age = ...      # TODO: max - min
range_income = ...   # TODO
print(range_age, range_income, round(range_income / range_age))

✅ **Self-check**

In [ ]:
assert range_age == 69
assert range_income / range_age > 2000, "income varies more than 2000 times as much as age"
print("✅ 2.1 correct")

✍️ **Question:** in a k-NN, the distance between two clients is $\sqrt{(\Delta \text{age})^2 + (\Delta \text{income})^2 + \dots}$. Which variable will **dominate** this distance? Why is that a problem?

…

### 2.2 Method 1 — Standardisation (Z-score)
$$z = \frac{x - \mu}{\sigma}$$
1. Compute `income_z` **by hand** with the formula (use `std(ddof=0)`, the population standard deviation, like scikit-learn).
2. Compute it with `StandardScaler` and check you get the same thing.

In [ ]:
income_z = ...         # TODO: formula by hand

scaler = ...           # TODO: StandardScaler()
income_z_sk = ...      # TODO: fit_transform on df[["income"]] (double brackets: a DataFrame) then .ravel()

print(round(income_z.mean(), 6), round(income_z.std(ddof=0), 6))

✅ **Self-check**

In [ ]:
assert np.allclose(income_z, income_z_sk), "the formula and StandardScaler must give the same result"
assert abs(income_z.mean()) < 1e-9 and abs(income_z.std(ddof=0) - 1) < 1e-9, "mean 0, standard deviation 1"
assert np.isclose(scaler.mean_[0], df["income"].mean()), "the scaler stores μ"
print("✅ 2.2 correct")

### 2.3 Method 2 — Min-Max normalisation
$$x' = \frac{x - x_{min}}{x_{max} - x_{min}} \in [0, 1]$$
Same exercise: by hand, then with `MinMaxScaler`.

In [ ]:
income_mm = ...        # TODO: formula by hand
income_mm_sk = ...     # TODO: MinMaxScaler().fit_transform(...).ravel()

✅ **Self-check**

In [ ]:
assert np.allclose(income_mm, income_mm_sk)
assert income_mm.min() == 0 and income_mm.max() == 1
print("✅ 2.3 correct")

### 2.4 Log transform: fixing the skewness
Income is **right-skewed** (seen in Lab 2). Z-score and Min-Max **do not change the shape** of the distribution; the logarithm does.

Compute `income_log = np.log1p(income)` ($\log(1+x)$, defined even for $x = 0$), then compare the **skewness** (`skew()`) before / after.

In [ ]:
income_log = ...   # TODO

skew = pd.Series({
    "raw":     ...,   # TODO: df["income"].skew()
    "Z-score": ...,
    "Min-Max": ...,
    "log":     ...,
}).round(2)
skew

✅ **Self-check**

In [ ]:
assert skew["raw"] == skew["Z-score"] == skew["Min-Max"], "Z-score and Min-Max keep the shape"
assert abs(skew["log"]) < 0.3 < skew["raw"], "the log makes the distribution almost symmetric"
print("✅ 2.4 correct")

Provided cell: the four versions of income side by side (as on the "Compare: Z-score vs Min-Max" slide).

In [ ]:
versions = {"Raw (DA)": df["income"], "Z-score": income_z, "Min-Max": income_mm, "log(1 + income)": income_log}
colours = ["gray", "tab:blue", "tab:green", "tab:purple"]

fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))
for ax, (title, series), c in zip(axes, versions.items(), colours):
    sns.histplot(series, bins=30, kde=True, color=c, ax=ax)
    ax.set_title(f"{title}\nskew = {series.skew():.2f}", fontsize=10)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

### 2.5 Sensitivity to extreme values
In Lab 2 we replaced the **999,999** income (a data-entry error) with an imputed value. What would have happened if we had **forgotten**?

On a copy of income where we put **one** value back to 999,999, compute the **median** of the values transformed by Min-Max and by Z-score, with and without the outlier.

In [ ]:
with_outlier = df["income"].copy()
with_outlier.iloc[0] = 999999

def minmax(s):
    return (s - s.min()) / (s.max() - s.min())

def zscore(s):
    return (s - s.mean()) / s.std(ddof=0)

effect = pd.DataFrame({
    "without outlier": ...,   # TODO: {"Min-Max median": ..., "Z-score median": ...} computed on df["income"]
    "with outlier": ...,      # TODO: same on with_outlier
}).round(3)
effect

✅ **Self-check**

In [ ]:
assert effect.loc["Min-Max median", "with outlier"] < 0.05 < effect.loc["Min-Max median", "without outlier"]
print("✅ 2.5 correct")

✍️ **Questions**
1. After Min-Max, what happens to the range occupied by the "normal" clients when the outlier is present?
2. According to the "Choice rule" slide, which method should be used for income? And for `age`?

1. …
2. …

### 2.6 Good practice: learn μ and σ on the training set only
*"Always compute μ and σ on the training set only. Apply the same transformation to validation, test and new data."*

1. Split `df` into training (80%) and test (20%) with `train_test_split(..., test_size=0.2, random_state=42, stratify=df["segment"])`.
2. `fit` a `StandardScaler` on `num_cols` of the **train set** only, then `transform` both train **and** test.

In [ ]:
df_train, df_test = ...   # TODO

scaler = StandardScaler()
# TODO: fit on the train set
X_train_num = ...   # TODO: transform the train set
X_test_num = ...    # TODO: transform the test set (without fitting again!)

print("train mean:", X_train_num.mean(axis=0).round(3))
print("test mean :", X_test_num.mean(axis=0).round(3))

✅ **Self-check**

In [ ]:
assert len(df_train) == 790 and len(df_test) == 198
assert np.allclose(X_train_num.mean(axis=0), 0), "the train set is centred"
assert not np.allclose(X_test_num.mean(axis=0), 0), "the test set must NOT be exactly centred"
assert np.allclose(scaler.mean_, df_train[num_cols].mean()), "μ learned on the train set only"
print("✅ 2.6 correct")

✍️ **Question:** why are the **test** means not exactly 0? Why must we **not** fit again on the test set?

…

---
## Step 3 — Encoding the categorical variables (section 7)
Reminder of the decision guide: **natural order?** no → **One-Hot**; yes → is it the **target**? yes → **Label**, no → **Ordinal**.

### 3.1 One-Hot Encoding — `city` (nominal)
1. Encode `city` with `pd.get_dummies(..., prefix="city", dtype=int)`.
2. Do it again with `drop_first=True`. How many columns in each case?

In [ ]:
city_oh = ...        # TODO
city_oh_df = ...     # TODO: with drop_first=True

print(city_oh.shape, city_oh_df.shape)
city_oh.head()

✅ **Self-check**

In [ ]:
assert city_oh.shape == (988, 5), "5 cities -> 5 binary columns"
assert (city_oh.sum(axis=1) == 1).all(), "exactly one 1 per row"
assert city_oh_df.shape == (988, 4)
print("✅ 3.1 correct")

✍️ **Questions**
1. With `drop_first=True`, which city disappeared? How do we still recognise a client from that city?
2. Why would it be **wrong** to encode `city` as Oran = 0, Alger = 1, Béchar = 2…?

1. …
2. …

### 3.2 Ordinal Encoding — `education` (ordinal)
1. First try `OrdinalEncoder()` **without giving the order** (cell provided): which code does each level get?
2. Fix it by giving the order explicitly: `categories=[["Bachelor", "Master", "Doctorate"]]`.

In [ ]:
trap = OrdinalEncoder().fit(df[["education"]])
print("Order learned by default:", list(trap.categories_[0]))

In [ ]:
enc_education = ...   # TODO: OrdinalEncoder with the correct order
education_code = ...  # TODO: fit_transform(...).ravel()

pd.crosstab(df["education"], education_code)

✅ **Self-check**

In [ ]:
codes = pd.Series(education_code, index=df.index).groupby(df["education"]).first()
assert codes["Bachelor"] < codes["Master"] < codes["Doctorate"], "the natural order must be respected"
assert df["income"].corr(pd.Series(education_code, index=df.index)) > 0.4, "education and income are linked"
print("✅ 3.2 correct")

✍️ **Question:** what did the default encoding give? Why is it serious here?

…

### 3.3 Label Encoding — `segment` (target)
Encode the target with `LabelEncoder` and display the **learned mapping** (`le.classes_`).

In [ ]:
le = ...   # TODO: LabelEncoder()
y = ...    # TODO: fit_transform on df["segment"]

mapping = dict(zip(le.classes_, le.transform(le.classes_)))
mapping

✅ **Self-check**

In [ ]:
assert list(le.classes_) == ["A", "B", "C"] and set(y) == {0, 1, 2}
assert (le.inverse_transform(y) == df["segment"]).all(), "we must be able to get the labels back"
print("✅ 3.3 correct")

### 3.4 Pitfall: an unknown category at prediction time
A new client arrives from **Tlemcen**, a city absent from the training data.
1. Provided cell: a default `OneHotEncoder` **crashes**.
2. Create `enc_city` with `handle_unknown="ignore"` and `sparse_output=False`, fit it on `df_train[["city"]]`, then encode the new client.

In [ ]:
new_client = pd.DataFrame({"city": ["Tlemcen"]})

try:
    OneHotEncoder().fit(df_train[["city"]]).transform(new_client)
except ValueError as e:
    print("❌ Error:", str(e)[:90], "...")

In [ ]:
enc_city = ...    # TODO
code_new = ...    # TODO: transform(new_client)

print(enc_city.get_feature_names_out(), code_new)

✅ **Self-check**

In [ ]:
assert code_new.shape == (1, 5) and code_new.sum() == 0, "unknown city -> all zeros"
print("✅ 3.4 correct")

---
## Step 4 — Assemble the preparation and measure its effect
In a real project, all transformations are grouped in a **`ColumnTransformer`**: each group of columns gets its own treatment, and the same object is applied to train and test (good practice ⑤: *same encoding for training and test*).

### 4.1 The `ColumnTransformer`
Complete `prep`: `StandardScaler` on `num_cols`, `OneHotEncoder` (unknowns ignored) on `nom_cols`, `OrdinalEncoder` with **both** orders on `ord_cols`. `id`, `email` and `segment` are not explanatory variables.

In [ ]:
from sklearn.compose import ColumnTransformer

orders = [["Bachelor", "Master", "Doctorate"], ["low", "medium", "high"]]

prep = ColumnTransformer([
    ("num", ..., num_cols),   # TODO
    ("nom", ..., nom_cols),   # TODO
    ("ord", ..., ord_cols),   # TODO
])

X_train = prep.fit_transform(df_train)   # fit on the train set
X_test = prep.transform(df_test)         # transform only on the test set
y_train, y_test = le.transform(df_train["segment"]), le.transform(df_test["segment"])

print(X_train.shape, X_test.shape)
pd.DataFrame(X_train, columns=prep.get_feature_names_out()).head()

✅ **Self-check**

In [ ]:
assert X_train.shape == (790, 12) and X_test.shape == (198, 12), "5 numeric + 5 cities + 2 ordinal = 12"
assert "email" not in " ".join(prep.get_feature_names_out()), "email is not an explanatory variable"
print("✅ 4.1 correct")

### 4.2 Does scaling change the model?
We predict the `segment` with a **k-NN** (k = 15), a distance-based model. Compare the test **accuracy**:
- **without scaling** the numeric variables (`"passthrough"` instead of the `StandardScaler`);
- with **Z-score**;
- with **Min-Max**.

💡 `Pipeline` chains the preparation and the model: `fit` on the train set, `score` on the test set.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier

def evaluate(scaling):
    """Test accuracy of k-NN (k=15) for a given scaling of the numeric variables."""
    prep = ColumnTransformer([
        ("num", scaling, num_cols),
        ("nom", OneHotEncoder(handle_unknown="ignore"), nom_cols),
        ("ord", OrdinalEncoder(categories=orders), ord_cols),
    ])
    model = Pipeline([("prep", prep), ("knn", KNeighborsClassifier(n_neighbors=15))])
    # TODO: fit on (df_train, y_train), return the score on (df_test, y_test)
    ...

results = pd.Series({
    "None": ...,      # TODO: evaluate("passthrough")
    "Z-score": ...,   # TODO
    "Min-Max": ...,   # TODO
}, name="accuracy").round(3)
results

✅ **Self-check**

In [ ]:
assert results["Z-score"] > results["None"] + 0.05, "standardisation must clearly improve the k-NN"
print("✅ 4.2 correct")

✍️ **Your analysis:** explain the gap between "None" and "Z-score" using what you saw in 2.1. Why does Min-Max do worse than Z-score here?

⭐ **Bonus:** add a `log1p` transform of income before the Z-score (hint: `FunctionTransformer(np.log1p)` and `make_pipeline`). Does the result change?

…

---
## Step 5 — Document every decision
*Good practice ⑥: keep the mapping used, document for reproducibility.*

✍️ **Decision log** — one row per variable (or group of variables).

| Variable(s) | Nature | Chosen treatment | Justification |
|---|---|---|---|
| `id`, `email` | … | … | … |
| `age`, `n_children`, `tenure`, `n_purchases` | … | … | … |
| `income` | … | … | … |
| `city` | … | … | … |
| `education`, `satisfaction` | … | … | … |
| `segment` | … | … | … |

✍️ **Conclusions** (5 to 10 lines): which scaling, which encodings, what effect was measured on the model, which pitfalls were avoided?

…

---
## Summary

| Need | Tool | Remember |
|---|---|---|
| Same scale, centred and scaled | `StandardScaler` | default method, less sensitive to outliers |
| Same scale, bounded [0, 1] | `MinMaxScaler` | very sensitive to extreme values |
| Fix skewness | `np.log1p` | changes the **shape**, not just the scale |
| Nominal (no order) | `OneHotEncoder` / `get_dummies` | `handle_unknown="ignore"`, `drop_first` for linear models |
| Ordinal (ordered) | `OrdinalEncoder(categories=...)` | always give the order explicitly |
| Target y | `LabelEncoder` | never for input variables X |
| Put it all together | `ColumnTransformer` + `Pipeline` | `fit` on train, `transform` on test |

**Golden rule:** the model must never interpret an order that does not really exist.